In [1]:
import dotenv
import os

dotenv.load_dotenv(dotenv_path='.env')

True

In [2]:
import pyarrow as pa
import pandas
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.exceptions import NamespaceAlreadyExistsError
from pyiceberg.transforms import IdentityTransform, DayTransform

/Users/jhonsfran/repos/unprice/internal/lakehouse/lakenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [23]:
# 1. Connect to R2 Data Catalog

# Define catalog connection details
ENV = "prod"
CATALOG = "unprice-lakehouse-" + ENV
ACCOUNT_ID  = os.environ['ACCOUNT_ID']
CATALOG_URI = "https://catalog.cloudflarestorage.com/" + ACCOUNT_ID + "/" + CATALOG
TOKEN       = os.environ['TOKEN']
WAREHOUSE   = ACCOUNT_ID + "_" + CATALOG

# Connect to R2 Data Catalog
catalog = RestCatalog(
    name="unprice",
    warehouse=WAREHOUSE,
    uri=CATALOG_URI,
    token=TOKEN,
)

# list namespaces
print(catalog.list_namespaces())

[('lakehouse',)]


In [25]:
# drop tables
# catalog.drop_table('lakehouse.metadata')
# catalog.drop_table('lakehouse.entitlement_snapshot')
# catalog.drop_table('lakehouse.usage')
# catalog.drop_table('lakehouse.verification')

In [26]:
# List tables in the "default" namespace
tables = catalog.list_tables(namespace='lakehouse')
print(tables)

[('lakehouse', 'events')]


In [27]:
# Load and print schema for every table in the lakehouse namespace
tables = catalog.list_tables(namespace='lakehouse')
for table_id in tables:
    table = catalog.load_table(table_id)
    print(f"--- {table_id[0]}.{table_id[1]} ---")
    print(table.schema())
    print()

--- lakehouse.events ---
table {
  1: __ingest_ts: required timestamp
  2: event_date: required timestamp
  3: schema_version: required int
  4: id: required string
  5: project_id: required string
  6: customer_id: required string
  7: request_id: required string
  8: idempotency_key: required string
  9: slug: required string
  10: timestamp: required timestamp
  11: received_at: required timestamp
  12: handled_at: required timestamp
  13: state: required string
  14: rejection_reason: optional string
  15: properties: required string
}



In [28]:
table = catalog.load_table("lakehouse.events")
table.refresh()  # <-- this is the key
df = table.scan().to_arrow()
print(df)

pyarrow.Table
__ingest_ts: timestamp[us] not null
event_date: timestamp[us] not null
schema_version: int32 not null
id: large_string not null
project_id: large_string not null
customer_id: large_string not null
request_id: large_string not null
idempotency_key: large_string not null
slug: large_string not null
timestamp: timestamp[us] not null
received_at: timestamp[us] not null
handled_at: timestamp[us] not null
state: large_string not null
rejection_reason: large_string
properties: large_string not null
----
__ingest_ts: [[]]
event_date: [[]]
schema_version: [[]]
id: [[]]
project_id: [[]]
customer_id: [[]]
request_id: [[]]
idempotency_key: [[]]
slug: [[]]
timestamp: [[]]
...


In [29]:
# List and peek contents of every table in the lakehouse namespace
import pandas as pd

MAX_ROWS = 1000  # rows to show per table

table = catalog.load_table("lakehouse.events")
scan = table.scan(limit=MAX_ROWS)
df = scan.to_pandas()
print(f"\n=== {name} (showing up to {MAX_ROWS} rows) ===")
if df.empty:
    print("(no rows)")
else:
    display(df)


=== lakehouse.usage (showing up to 1000 rows) ===
(no rows)


In [230]:
from pyiceberg.expressions import And, EqualTo

# Load your table
table = catalog.load_table("lakehouse.verification")  # adjust namespace.table_name
table.refresh()

seven_days_ago = (datetime.utcnow() - timedelta(days=7)).isoformat()

# Build the expression tree
filter_expr = And(
    GreaterThanOrEqual("__ingest_ts", seven_days_ago),
    EqualTo("customer_id", "cus_11TqNF6bCebUjnx55pk6vs"),
    EqualTo("project_id", "proj_11STWG6AokEni2F3eQugHb")
)

# PyIceberg will use this to prune partitions/files and return only the matching rows
scan = table.scan(
    row_filter=filter_expr
)

# Get all matching files
files = []
for task in scan.plan_files():
    files.append({
        "file_path": task.file.file_path,
        "record_count": task.file.record_count,
        "file_size_in_bytes": task.file.file_size_in_bytes,
    })

print(f"Found {len(files)} files with {sum(f['record_count'] for f in files)} total records")
for f in files:
    print(f["file_path"])

Found 1 files with 28 total records
s3://unprice-lakehouse-dev/__r2_data_catalog/019c66ef-2bb2-7eb0-ba6f-40471a747007/019c80ad-4529-7ed2-9455-f5c36c5ed590/data/8eb4f53b-2923-4ca0-9789-48734a9b4f5f.parquet


In [226]:
from pyiceberg.expressions import GreaterThanOrEqual, EqualTo, And
from datetime import datetime, timedelta

# Calculate 7 days ago dynamically
seven_days_ago = (datetime.utcnow() - timedelta(days=7)).isoformat()

# Build the expression tree
filter_expr = And(
    GreaterThanOrEqual("__ingest_ts", seven_days_ago),
    EqualTo("customer_id", "cus_11TqNF6bCebUjnx55pk6vs")
)

# PyIceberg will use this to prune partitions/files and return only the matching rows
scan = table.scan(
    row_filter=filter_expr
)

arrow_table = scan.to_arrow()

pyarrow.Table
__ingest_ts: timestamp[us] not null
event_date: timestamp[us] not null
id: string not null
project_id: string not null
entitlement_id: string not null
feature_slug: string not null
customer_id: string not null
request_id: string not null
timestamp: timestamp[us] not null
schema_version: int32 not null
allowed: bool
denied_reason: string
latency: double
country: string
region: string
meta_id: string
action: string
key_id: string
usage: double
remaining: int64
unit_of_measure: string
cost: double
rate_amount: double
rate_currency: string
----
__ingest_ts: [[2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,...,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000]]
event_date: [[2026-02-21 00:00:00.000000,2026-02-21 00:00:00.000000,2026-02-21 00:00:00.000000,2026-02-21 00:00:00.000000,2026-02-21 00:00:00.000000

In [21]:
# 1. Create namespace
# catalog.create_namespace("default")

In [190]:
# 2. Load the pipeline-created table
# verifications = catalog.load_table("lakehouse.verification")
# print("Current Spec:", verifications.spec())
# metadata = catalog.load_table("lakehouse.metadata")
# print("Current Spec:", verifications.spec())
# entitlement_snapshot = catalog.load_table("lakehouse.entitlement_snapshot")
# print("Current Spec:", verifications.spec())
usage = catalog.load_table("default.usage3")
print("Current Spec:", usage.spec())
print(usage.metadata.location)

Current Spec: [
  1000: __ingest_ts_day: day(1)
  1001: project_partition: identity(5)
  1002: customer_partition: identity(6)
]
s3://sapohpta/__r2_data_catalog/019c8005-2735-7211-9fe1-0224dba747b7/019c8062-b4ce-7430-a948-5248d4602077


In [185]:
old_partition_name = usage.spec().fields[0].name 
print(old_partition_name)

__ingest_ts_day


In [169]:
# verifications
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.verification")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

NoSuchTableError: TableActionForbidden: Table not found or action can_get_metadata forbidden for Anonymous

In [187]:
# usage
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("default.usage3")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # Remove the existing ingestion-time partition
    # update.remove_field(old_partition_name)

    # update.add_field("event_date", DayTransform(), old_partition_name)
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())


New Spec: [
  1000: __ingest_ts_day: day(1)
  1001: project_partition: identity(5)
  1002: customer_partition: identity(6)
]

Partition Data Preview:
                                           partition  spec_id  record_count  \
0  {'__ingest_ts_day': 2026-02-21, 'project_parti...        0            65   

   file_count  total_data_file_size_in_bytes  position_delete_record_count  \
0           1                          23438                             0   

   position_delete_file_count  equality_delete_record_count  \
0                           0                             0   

   equality_delete_file_count         last_updated_at  \
0                           0 2026-02-21 13:33:13.939   

   last_updated_snapshot_id  
0       2355266111474742490  


In [ ]:
# entitlement_snapshot
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.entitlement_snapshot")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

In [101]:
# metadata
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.metadata")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

New Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(5)
]

Partition Data Preview:


ValueError: Cannot get a snapshot as the table does not have any.

In [102]:
# 4. Load the pipeline-created table
verifications = catalog.load_table("lakehouse.verification")
print("Current Spec:", verifications.spec())
metadata = catalog.load_table("lakehouse.metadata")
print("Current Spec:", verifications.spec())
entitlement_snapshot = catalog.load_table("lakehouse.entitlement_snapshot")
print("Current Spec:", verifications.spec())
usage = catalog.load_table("lakehouse.usage")
print("Current Spec:", verifications.spec())
print(verifications.metadata.location)

Current Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(7)
]
Current Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(7)
]
Current Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(7)
]
Current Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(7)
]
s3://unprice-dev/__r2_data_catalog/019c7d38-b6b3-7c90-94ee-da8a2e523815/019c7d3b-52be-7790-ba1a-c2abb7ca1803
